# L17 – Regression: Fitting Models to Stellar Data

In this notebook we apply the regression techniques from `L17_regression.ipynb` to the `stars.csv` dataset.

We will:
1. Load and explore the stellar dataset
2. Fit a **linear regression** model
3. Fit **polynomial regression** models of increasing degree
4. Use **cross-validation** to select the best polynomial degree
5. Analyse over/underfitting via learning curves

---
## 0 – Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

---
## 1 – Load and explore the data

The `stars.csv` file contains physical properties of stars:

| Column | Unit |
|---|---|
| Temperature (K) | Kelvin |
| Luminosity(L/Lo) | Solar luminosities |
| Radius(R/Ro) | Solar radii |
| Absolute magnitude(Mv) | mag |
| Star type | categorical |
| Star color | categorical |
| Spectral Class | categorical |

We will model the relationship between **Temperature** (predictor $x$) and **Luminosity** (response $y$), 
which is related to the [Hertzsprung-Russell diagram](https://en.wikipedia.org/wiki/Hertzsprung%E2%80%93Russell_diagram).

In [ ]:
# Load the data – path relative to the working/ directory
df = pd.read_csv('../stars.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

In [ ]:
# Work on log scale – both quantities span many orders of magnitude
log_T   = np.log10(df['Temperature (K)'].values)
log_L   = np.log10(df['Luminosity(L/Lo)'].values)

plt.figure()
plt.scatter(log_T, log_L, s=20, alpha=0.6, edgecolors='k', linewidths=0.3)
plt.xlabel(r'$\log_{10}(T \; [\mathrm{K}])$')
plt.ylabel(r'$\log_{10}(L / L_\odot)$')
plt.title('Hertzsprung–Russell diagram (log–log)')
plt.gca().invert_xaxis()   # hot stars on the left, cool on the right
plt.tight_layout()
plt.show()

---
## 2 – Linear Regression (polynomial degree 1)

We fit the simplest model: a straight line in log–log space.

$$\log L = a_0 + a_1 \log T$$

This is equivalent to the power-law $L \propto T^{a_1}$.

In [ ]:
# --- numpy polyfit approach (mirrors L17) ---
x = log_T
y = log_L

coeffs = np.polyfit(x, y, deg=1)   # [slope, intercept]
print(f'Linear fit:  slope = {coeffs[0]:.3f},  intercept = {coeffs[1]:.3f}')

x_fit = np.linspace(x.min(), x.max(), 300)
y_fit = np.polyval(coeffs, x_fit)

plt.figure()
plt.scatter(x, y, s=20, alpha=0.5, label='data')
plt.plot(x_fit, y_fit, 'r-', lw=2, label=f'Linear fit (d=1)')
plt.xlabel(r'$\log_{10}(T)$')
plt.ylabel(r'$\log_{10}(L/L_\odot)$')
plt.title('Linear regression on stellar data')
plt.legend()
plt.tight_layout()
plt.show()

# RMS residual
y_pred = np.polyval(coeffs, x)
rms = np.sqrt(np.mean((y_pred - y)**2))
print(f'RMS residual (training): {rms:.4f}')

The scatter around the best-fit line is large because the HR diagram has distinct stellar populations (main sequence, giants, dwarfs, …) which a single straight line cannot capture well.

---
## 3 – Polynomial Regression

We now try polynomial models of degree $d = 1, 2, 3, \ldots, 10$ and see how they fit.

$$\log L = a_0 + a_1 \log T + a_2 (\log T)^2 + \ldots + a_d (\log T)^d$$

In [ ]:
degrees = [1, 2, 3, 5, 8, 10]

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, d in zip(axes, degrees):
    coeffs = np.polyfit(x, y, deg=d)
    y_pred_all = np.polyval(coeffs, x)
    rms = np.sqrt(np.mean((y_pred_all - y)**2))

    ax.scatter(x, y, s=10, alpha=0.4, label='data')
    ax.plot(x_fit, np.polyval(coeffs, x_fit), 'r-', lw=2)
    ax.set_title(f'd = {d}   (RMS = {rms:.3f})')
    ax.set_xlabel(r'$\log_{10}(T)$')
    ax.set_ylabel(r'$\log_{10}(L/L_\odot)$')

plt.suptitle('Polynomial regression — increasing degree', fontsize=14)
plt.tight_layout()
plt.show()

As $d$ grows the polynomial passes closer and closer to the training points (training RMS decreases), but high-degree polynomials can **overfit** – they may behave badly on unseen data.

---
## 4 – Cross-validation to select the best degree

We split the data into **training** and **cross-validation** sets (80 / 20 split), then compute both training and CV RMS as a function of the polynomial degree.

In [ ]:
np.random.seed(42)
x_train, x_cv, y_train, y_cv = train_test_split(x, y, test_size=0.20, random_state=42)

max_degree = 15
degrees_all = np.arange(1, max_degree + 1)

train_rms = np.zeros(max_degree)
cv_rms    = np.zeros(max_degree)

for i, d in enumerate(degrees_all):
    coeffs = np.polyfit(x_train, y_train, deg=d)
    train_rms[i] = np.sqrt(np.mean((np.polyval(coeffs, x_train) - y_train)**2))
    cv_rms[i]    = np.sqrt(np.mean((np.polyval(coeffs, x_cv)    - y_cv   )**2))

best_d = degrees_all[np.argmin(cv_rms)]
print(f'Best polynomial degree (CV): d = {best_d}  (CV RMS = {cv_rms[best_d-1]:.4f})')

plt.figure()
plt.plot(degrees_all, train_rms, 'b-o', label='Training RMS')
plt.plot(degrees_all, cv_rms,    'r-o', label='Cross-validation RMS')
plt.axvline(best_d, color='green', ls='--', label=f'Best d = {best_d}')
plt.xlabel('Polynomial degree $d$')
plt.ylabel('RMS error')
plt.title('Bias–variance trade-off')
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation:**
- For small $d$ both training and CV errors are high → *underfitting* (high bias).
- As $d$ increases the training error keeps dropping, but the CV error has a minimum and then starts rising again → *overfitting* (high variance).
- The optimal degree minimises the CV error.

---
## 5 – Best-fit polynomial

Show the polynomial of the optimal degree together with the data.

In [ ]:
coeffs_best = np.polyfit(x_train, y_train, deg=best_d)

plt.figure()
plt.scatter(x_train, y_train, s=15, alpha=0.5, label='Training data')
plt.scatter(x_cv,    y_cv,    s=15, alpha=0.5, marker='^', label='CV data')
plt.plot(x_fit, np.polyval(coeffs_best, x_fit), 'r-', lw=2, label=f'Poly d={best_d}')
plt.xlabel(r'$\log_{10}(T)$')
plt.ylabel(r'$\log_{10}(L/L_\odot)$')
plt.title(f'Best polynomial fit (d={best_d})')
plt.legend()
plt.tight_layout()
plt.show()

---
## 6 – Learning curves

For the best degree, how does the error change as we add more training data?  
This is the **learning curve** – it tells us whether more data would help.

In [ ]:
d_fixed = best_d

N_values = np.arange(d_fixed + 2, len(x_train) + 1, 5)   # need at least d+1 points
lc_train = np.zeros(len(N_values))
lc_cv    = np.zeros(len(N_values))

for i, N in enumerate(N_values):
    coeffs_lc = np.polyfit(x_train[:N], y_train[:N], deg=d_fixed)
    lc_train[i] = np.sqrt(np.mean((np.polyval(coeffs_lc, x_train[:N]) - y_train[:N])**2))
    lc_cv[i]    = np.sqrt(np.mean((np.polyval(coeffs_lc, x_cv)        - y_cv       )**2))

plt.figure()
plt.plot(N_values, lc_train, 'b-', label='Training RMS')
plt.plot(N_values, lc_cv,    'r-', label='CV RMS')
plt.xlabel('Number of training samples')
plt.ylabel('RMS error')
plt.title(f'Learning curve  (d = {d_fixed})')
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation:**
- If the training and CV curves converge to a similar value with many samples → adding more data will not help much (the model is probably a good match for the signal complexity).
- If the CV curve is still dropping steeply when we reach all available data → more data *would* help.

---
## 7 – Sklearn pipeline (alternative approach)

The same analysis can be done cleanly with `scikit-learn` `Pipeline`s, which is useful for more complex workflows.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge

X = x.reshape(-1, 1)
X_fit = x_fit.reshape(-1, 1)

cv_scores = []
degrees_sk = range(1, 16)
for d in degrees_sk:
    pipe = Pipeline([
        ('poly',  PolynomialFeatures(degree=d)),
        ('scale', StandardScaler()),
        ('reg',   LinearRegression())
    ])
    # 5-fold cross-validation; neg_root_mean_squared_error => negate to get positive RMS
    scores = cross_val_score(pipe, X, y, cv=5,
                             scoring='neg_root_mean_squared_error')
    cv_scores.append(-scores.mean())

best_d_sk = degrees_sk[np.argmin(cv_scores)]
print(f'Best degree (5-fold CV, sklearn): d = {best_d_sk}  (CV RMS = {cv_scores[best_d_sk-1]:.4f})')

plt.figure()
plt.plot(list(degrees_sk), cv_scores, 'go-')
plt.axvline(best_d_sk, color='red', ls='--', label=f'Best d={best_d_sk}')
plt.xlabel('Polynomial degree')
plt.ylabel('Mean 5-fold CV RMS')
plt.title('Sklearn 5-fold cross-validation')
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

| Technique | Key idea | Tool |
|---|---|---|
| Linear regression (d=1) | Straight line in feature space | `np.polyfit` |
| Polynomial regression (d>1) | Curved fit via higher-order features | `np.polyfit` / sklearn `PolynomialFeatures` |
| Cross-validation | Estimate generalisation error to choose degree | manual train/CV split or sklearn `cross_val_score` |
| Learning curves | Understand whether more data helps | plot error vs. $N_{\rm train}$ |

The optimal polynomial degree balances **bias** (underfitting) and **variance** (overfitting), and is best chosen by minimising the cross-validation error rather than the training error.